# 🏋️ DETECTOR DE FLEXIONES EN TIEMPO REAL
## Versión para Jupyter Notebook - Anaconda Ubuntu

### 📋 Instrucciones:
1. Ejecuta las celdas en orden
2. Colócate de **perfil** a la cámara
3. Para detener: **Interrumpe el kernel** (botón ⏹ o Kernel → Interrupt)

## 📦 IMPORTAR LIBRERÍAS

In [21]:
import cv2
from ultralytics import YOLO
import math
import numpy as np
from IPython.display import display, Image as IPImage, clear_output
import ipywidgets as widgets
from io import BytesIO
from PIL import Image

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 🔧 FUNCIONES AUXILIARES

In [22]:
def calcular_angulo(A, B, C):
    """
    Calcula el ángulo en el punto B formado por los puntos A-B-C
    """
    radianes = math.atan2(C[1] - B[1], C[0] - B[0]) - \
               math.atan2(A[1] - B[1], A[0] - B[0])
    angulo = abs(radianes * 180.0 / math.pi)
    if angulo > 180.0:
        angulo = 360 - angulo
    return angulo


def verificar_alineacion(hombro, cadera, tobillo):
    """
    Verifica que el cuerpo esté alineado (espalda recta)
    """
    angulo_espalda = calcular_angulo(tobillo, cadera, hombro)
    return 150 <= angulo_espalda <= 200


def verificar_caderas(cadera_y, hombro_y):
    """
    Verifica que las caderas no estén muy arriba o muy abajo
    """
    diferencia = abs(cadera_y - hombro_y)
    return diferencia < 150


print("✅ Funciones auxiliares definidas")

✅ Funciones auxiliares definidas


## 🎯 CARGAR MODELO YOLO

In [23]:
print("🔄 Cargando modelo YOLO...")
model = YOLO("yolo11s-pose.pt")
print("✅ Modelo cargado correctamente")

🔄 Cargando modelo YOLO...
✅ Modelo cargado correctamente


## 📹 DETECTOR PRINCIPAL - VERSIÓN JUPYTER

In [24]:
def detector_flexiones_jupyter():
    """
    Detector de flexiones optimizado para Jupyter Notebook
    Para detener: Interrumpe el kernel (Kernel → Interrupt)
    """
    
    # Abrir webcam
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("❌ Error: No se pudo abrir la webcam")
        print("💡 Verifica que la cámara esté conectada")
        print("💡 Prueba ejecutar: ls -l /dev/video*")
        return
    
    # Configurar resolución
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    # Variables de estado
    contador_correctas = 0
    contador_incorrectas = 0
    estado = None
    estado_ant = None
    
    print("🎥 Webcam iniciada")
    print("📌 Colócate de PERFIL a la cámara")
    print("⏹  Para detener: Kernel → Interrupt")
    print()
    
    frame_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("⚠️  No se pudo leer frame")
                break
            
            frame_count += 1
            
            # Procesar cada 2 frames para mejor rendimiento
            if frame_count % 2 != 0:
                continue
            
            # Efecto espejo
            frame = cv2.flip(frame, 1)
            
            # Detectar pose
            results = model(frame, verbose=False)
            
            # Variables de feedback
            errores = []
            flexion_correcta = True
            
            for r in results:
                if r.keypoints is None or r.keypoints.xy.shape[0] == 0:
                    continue
                
                kpts = r.keypoints.xy[0]
                conf = r.keypoints.conf[0]
                
                # Índices de keypoints
                IDX_HOMBRO = 6
                IDX_CODO = 8
                IDX_MUNECA = 10
                IDX_CADERA = 12
                IDX_RODILLA = 14
                IDX_TOBILLO = 16
                
                puntos = [IDX_HOMBRO, IDX_CODO, IDX_MUNECA, IDX_CADERA, IDX_RODILLA, IDX_TOBILLO]
                
                if all(conf[idx] > 0.5 for idx in puntos):
                    
                    # Extraer coordenadas
                    hombro = (int(kpts[IDX_HOMBRO][0]), int(kpts[IDX_HOMBRO][1]))
                    codo = (int(kpts[IDX_CODO][0]), int(kpts[IDX_CODO][1]))
                    muneca = (int(kpts[IDX_MUNECA][0]), int(kpts[IDX_MUNECA][1]))
                    cadera = (int(kpts[IDX_CADERA][0]), int(kpts[IDX_CADERA][1]))
                    rodilla = (int(kpts[IDX_RODILLA][0]), int(kpts[IDX_RODILLA][1]))
                    tobillo = (int(kpts[IDX_TOBILLO][0]), int(kpts[IDX_TOBILLO][1]))
                    
                    # ── ANÁLISIS ──────────────────────────────────────────────
                    
                    # 1. Ángulo del codo
                    angulo_codo = calcular_angulo(hombro, codo, muneca)
                    
                    # 2. Alineación corporal
                    alineacion_correcta = verificar_alineacion(hombro, cadera, tobillo)
                    
                    # 3. Posición de caderas
                    caderas_correctas = verificar_caderas(cadera[1], hombro[1])
                    
                    # 4. Rodillas extendidas
                    angulo_rodilla = calcular_angulo(cadera, rodilla, tobillo)
                    rodillas_extendidas = angulo_rodilla > 140
                    
                    # ── DETERMINAR ESTADO ─────────────────────────────────────
                    
                    if angulo_codo > 160:
                        estado = "arriba"
                    elif angulo_codo < 90:
                        estado = "abajo"
                    
                    # ── VALIDAR FORMA ─────────────────────────────────────────
                    
                    flexion_correcta = True
                    
                    if not alineacion_correcta:
                        errores.append("¡Espalda recta!")
                        flexion_correcta = False
                    
                    if not caderas_correctas:
                        errores.append("¡Caderas alineadas!")
                        flexion_correcta = False
                    
                    if not rodillas_extendidas:
                        errores.append("¡Extiende las piernas!")
                        flexion_correcta = False
                    
                    # ── CONTAR REPETICIONES ───────────────────────────────────
                    
                    if estado_ant == "abajo" and estado == "arriba":
                        if flexion_correcta:
                            contador_correctas += 1
                            print(f"✅ Flexión CORRECTA #{contador_correctas}")
                        else:
                            contador_incorrectas += 1
                            print(f"❌ Flexión INCORRECTA ({', '.join(errores)})")
                    
                    estado_ant = estado
                    
                    # ── COLORES ───────────────────────────────────────────────
                    
                    if flexion_correcta:
                        color_principal = (0, 255, 0)  # Verde
                        color_texto = (0, 255, 0)
                    else:
                        color_principal = (0, 0, 255)  # Rojo
                        color_texto = (0, 0, 255)
                    
                    if angulo_codo > 160:
                        color_codo = (0, 255, 0)
                    elif angulo_codo < 90:
                        color_codo = (255, 0, 0)
                    else:
                        color_codo = (0, 255, 255)
                    
                    # ── DIBUJAR SKELETON ──────────────────────────────────────
                    
                    # Brazo
                    cv2.line(frame, hombro, codo, color_principal, 4)
                    cv2.line(frame, codo, muneca, color_principal, 4)
                    
                    # Torso
                    color_torso = (0, 255, 0) if alineacion_correcta else (0, 0, 255)
                    cv2.line(frame, hombro, cadera, color_torso, 4)
                    
                    # Piernas
                    color_piernas = (0, 255, 0) if rodillas_extendidas else (0, 0, 255)
                    cv2.line(frame, cadera, rodilla, color_piernas, 4)
                    cv2.line(frame, rodilla, tobillo, color_piernas, 4)
                    
                    # Círculos en articulaciones
                    cv2.circle(frame, hombro, 8, (255, 0, 0), -1)
                    cv2.circle(frame, codo, 8, color_codo, -1)
                    cv2.circle(frame, muneca, 8, (255, 0, 0), -1)
                    cv2.circle(frame, cadera, 8, (255, 255, 0), -1)
                    cv2.circle(frame, rodilla, 8, (255, 255, 0), -1)
                    cv2.circle(frame, tobillo, 8, (255, 255, 0), -1)
                    
                    # ── TEXTO ─────────────────────────────────────────────────
                    
                    cv2.putText(frame, f"{int(angulo_codo)}°",
                               (codo[0] - 40, codo[1] - 15),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_codo, 2)
                    
                    cv2.putText(frame, f"Estado: {estado if estado else 'Posicionate'}",
                               (20, 50),
                               cv2.FONT_HERSHEY_SIMPLEX, 1.0, color_texto, 2)
            
            # ── PANEL DE INFORMACIÓN ──────────────────────────────────────────
            
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 80), (500, 300), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
            
            cv2.putText(frame, f"Correctas: {contador_correctas}",
                       (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
            
            cv2.putText(frame, f"Incorrectas: {contador_incorrectas}",
                       (20, 170), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            
            total = contador_correctas + contador_incorrectas
            cv2.putText(frame, f"Total: {total}",
                       (20, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
            
            # Mostrar errores o éxito
            if errores:
                y_pos = 270
                for error in errores:
                    cv2.putText(frame, f"⚠ {error}",
                               (20, y_pos),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    y_pos += 35
            else:
                cv2.putText(frame, "✓ Forma correcta!",
                           (20, 270),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            
            # ── MOSTRAR EN JUPYTER ────────────────────────────────────────────
            
            # Convertir BGR a RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Codificar a JPEG
            _, buffer = cv2.imencode('.jpg', frame_rgb)
            
            # Limpiar output anterior y mostrar nuevo frame
            clear_output(wait=True)
            display(IPImage(data=buffer.tobytes()))
    
    except KeyboardInterrupt:
        print("\n⏸  Detenido por el usuario")
    
    finally:
        # Liberar recursos
        cap.release()
        
        # Mostrar resumen final
        clear_output(wait=True)
        
        print("\n" + "="*60)
        print("📊 RESUMEN FINAL")
        print("="*60)
        print(f"✅ Flexiones correctas:   {contador_correctas}")
        print(f"❌ Flexiones incorrectas: {contador_incorrectas}")
        print(f"📈 Total:                 {total}")
        
        if total > 0:
            porcentaje = (contador_correctas / total) * 100
            print(f"🎯 Precisión:             {porcentaje:.1f}%")
        
        print("="*60)


print("✅ Función detector_flexiones_jupyter() definida")
print("\n💡 Para ejecutar: detector_flexiones_jupyter()")

✅ Función detector_flexiones_jupyter() definida

💡 Para ejecutar: detector_flexiones_jupyter()


## 🚀 EJECUTAR DETECTOR

### ⚠️ INSTRUCCIONES:
1. Ejecuta la celda siguiente
2. Verás el video en tiempo real debajo
3. Colócate de **perfil** a la cámara
4. Para detener: **Kernel → Interrupt** (o botón ⏹)
5. El resumen aparecerá automáticamente al detener

In [27]:
# EJECUTAR DETECTOR
detector_flexiones_jupyter()


📊 RESUMEN FINAL
✅ Flexiones correctas:   5
❌ Flexiones incorrectas: 3
📈 Total:                 8
🎯 Precisión:             62.5%


---

## 🔧 SOLUCIÓN DE PROBLEMAS

### ❌ Si la webcam no se abre:

```bash
# En terminal:
ls -l /dev/video*
sudo chmod 666 /dev/video0
```

### ❌ Si hay error de permisos:

```bash
# Instalar dependencias:
sudo apt-get install libgtk2.0-dev pkg-config
```

### 🔍 Verificar cámaras disponibles:

In [26]:
# EJECUTAR ESTA CELDA PARA VERIFICAR CÁMARAS
print("🔍 Buscando cámaras disponibles...\n")

for i in range(5):
    cap = cv2.VideoCapture(i)
    if cap.isOpened():
        ret, frame = cap.read()
        if ret:
            print(f"✅ Cámara {i}: Disponible ({frame.shape[1]}x{frame.shape[0]})")
        cap.release()

print("\n✅ Verificación completada")

🔍 Buscando cámaras disponibles...


✅ Verificación completada


[ WARN:0@2187.223] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@2187.223] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@2187.223] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[ERROR:0@2187.223] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@2187.223] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video2): can't open camera by index
[ERROR:0@2187.223] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@2187.223] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video3): can't open camera by index
[ERROR:0@2187.223] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@2187.223] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video4): can't open camera by index
[ERROR:0@2187.223] global obsensor_uvc_stream_channel.c

---

## 📝 NOTAS

### 🎯 Validaciones que realiza:
1. ✅ **Ángulo del codo** (debe bajar de 90° y subir a 160°)
2. ✅ **Espalda recta** (alineación hombro-cadera-tobillo)
3. ✅ **Caderas alineadas** (no muy arriba ni muy abajo)
4. ✅ **Piernas extendidas** (rodillas con ángulo > 160°)

### 🎨 Código de colores:
- 🟢 **Verde**: Forma correcta
- 🔴 **Rojo**: Error en la forma
- 🟡 **Amarillo**: Posición intermedia

### ⌨️ Controles:
- **Kernel → Interrupt**: Detener el detector
- Al detener se muestra el resumen automáticamente

---

**Creado por:** Manuel Llano  
**Fecha:** 2025  
**Entorno:** Anaconda + Ubuntu + Jupyter Notebook